# 01.06 · Tokens, cost, and rate limits

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 01.01–01.05 (especially batching). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Read** `usage_metadata` (input, output, reasoning and cached tokens) and explain which ones you pay for.
2. **Aggregate** usage across many calls and models with `get_usage_metadata_callback`, and **estimate cost** with the course price table (`mastery.pricing`).
3. **Read** Groq's rate-limit headers and explain RPM/RPD/TPM/TPD.
4. **Throttle** requests with `InMemoryRateLimiter`, and explain what it cannot do.
5. **Handle** HTTP 429: what the built-in retries do (you'll read the SDK code), `with_retry`, and exponential backoff with jitter that respects `retry-after`.

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [Models → Token usage](https://docs.langchain.com/oss/python/langchain/models#token-usage) · [Rate limiting](https://docs.langchain.com/oss/python/langchain/models#rate-limiting) | **Main page.** Callback/context manager, `InMemoryRateLimiter` |
| [`UsageMetadata` reference](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata) · [`InMemoryRateLimiter` reference](https://reference.langchain.com/python/langchain-core/rate_limiters/InMemoryRateLimiter) | Field definitions; limiter parameters |
| [Groq rate limits](https://console.groq.com/docs/rate-limits) | Limit types, headers, organization-level limits |
| [Groq API errors](https://console.groq.com/docs/errors) | 429, 498, 5xx and what to do |
| [LangSmith cost tracking](https://docs.langchain.com/langsmith/cost-tracking) | How LangSmith computes cost from traces (Lesson 01.08, Module 10) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "pandas>=3.0.6,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Tokens in one minute

Models read and write **tokens**: chunks of text, often a word piece. A rough rule for English is **~4 characters per token** (~0.75 words); other languages and code usually need more tokens per word. Providers bill and rate-limit by tokens:

| Token type | What it is | Billed as |
|---|---|---|
| **Input** (prompt) | System prompt + history + user message + tool schemas + images | Input price |
| **Output** (completion) | Everything the model generates | Output price (usually 3–5× input) |
| **Reasoning** | The thinking part of the output (`gpt-oss`, Qwen) | **Included in output**: billed as output |
| **Cached input** | Repeated prompt prefixes the provider recognized | Often discounted; on Groq, not counted toward rate limits ([rate limits page](https://console.groq.com/docs/rate-limits)) |

Exact counts need the model's own tokenizer. LangChain's generic `get_num_tokens` falls back to a GPT-2 tokenizer (and needs `transformers`), which is wrong for these models. **Course rule:** estimate with characters ÷ 4 *before* a call, and trust `usage_metadata` *after*.

In [ ]:
# 🧪 Offline: the characters/4 estimate. Good enough for budgeting, never for billing.
def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English text."""
    return max(1, round(len(text) / 4))

for sample in ["Hello!", "Explain rate limits in two sentences.", "¿Cuál es la capital de Colombia?", "def f(x): return x**2"]:
    print(f"{estimate_tokens(sample):>3} ≈ tokens  ← {sample!r}")

---
## Part 2 · Reading `usage_metadata`

Every `AIMessage` from a provider that reports usage has a standard `usage_metadata` dict ([`UsageMetadata`](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata)): `input_tokens`, `output_tokens`, `total_tokens`, and optional details such as `input_token_details.cache_read` and `output_token_details.reasoning`. Same keys for every provider; the raw provider numbers stay in `response_metadata`.

In [ ]:
# 💸 Calls Groq (fast model), ~300 tokens. Read usage and estimate cost.
from mastery.models import get_model
from mastery.pricing import estimate_cost, get_price

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm = get_model("fast", temperature=0, max_tokens=400, reasoning_effort="low")
    reply = llm.invoke("In two sentences, what is a rate limit?")
    usage = reply.usage_metadata
    print("usage_metadata:", usage)
    reasoning = usage.get("output_token_details", {}).get("reasoning", 0)
    print(f"answer tokens ≈ {usage['output_tokens'] - reasoning}, reasoning tokens = {reasoning} (both billed as output)")
    model_key = f"groq:{settings.model_fast}"
    print(f"price: {get_price(model_key)}")
    print(f"estimated cost of this call: ${estimate_cost(usage, model_key):.6f}")

### What just happened?

- With a key: something like `input_tokens ≈ 90`, `output_tokens ≈ 100`, of which a good part is **reasoning**. Reasoning is often the biggest share of output on small questions: `reasoning_effort="low"` is a real cost lever.
- The estimate is tiny (fractions of a cent) and on the free tier you pay nothing. The point is the *habit*: know what a call would cost on a paid tier before you run it a million times.
- `mastery.pricing` keeps prices in one dated table ([`src/mastery/pricing.py`](../../src/mastery/pricing.py)), with a `verified` date and a source URL per row (Lesson 00.07's rule: prices go stale).

---
## Part 3 · Aggregating usage across calls and models

For a whole notebook, job or agent run, collect usage with **`get_usage_metadata_callback()`**, a context manager that sums `usage_metadata` **per model name** for every call made inside the `with` block ([Models → Token usage](https://docs.langchain.com/oss/python/langchain/models#token-usage)). Feed the result to `mastery.pricing.cost_report` for a cost table. Offline first, with fake models that report usage.

In [ ]:
# 🧪 Offline: two fake "models" reporting usage under real model names, aggregated by the callback.
import pandas as pd
from IPython.display import display
from langchain.messages import AIMessage
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

from mastery.pricing import cost_report

def fake_reply(model_name: str, inp: int, out: int) -> AIMessage:
    return AIMessage("ok", response_metadata={"model_name": model_name},
                     usage_metadata={"input_tokens": inp, "output_tokens": out, "total_tokens": inp + out})

fast_fake = GenericFakeChatModel(messages=iter([fake_reply("openai/gpt-oss-20b", 100, 50)] * 3))
big_fake = GenericFakeChatModel(messages=iter([fake_reply("openai/gpt-oss-120b", 400, 300)]))
with get_usage_metadata_callback() as usage_cb:
    for _ in range(3):
        fast_fake.invoke("hi")
    big_fake.invoke("hi")
display(pd.DataFrame(cost_report(usage_cb.usage_metadata)))

In [ ]:
# 💸 Calls Groq 4 times (3 fast + 1 reasoning), ~1.2k tokens. Real aggregation.
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    thinker = get_model("reasoning", temperature=0, max_tokens=600, reasoning_effort="low")
    with get_usage_metadata_callback() as live_cb:
        llm.batch(["Define RPM.", "Define TPM.", "Define TPD."], config={"max_concurrency": 3})
        thinker.invoke("Why do providers limit tokens per minute, not just requests? Two sentences.")
    display(pd.DataFrame(cost_report(live_cb.usage_metadata)))

### What just happened?

- The offline table shows `openai/gpt-oss-20b` with 300 input / 150 output tokens (3 calls summed) and `openai/gpt-oss-120b` with 400 / 300, each with a cost and a `price_verified` date.
- The callback keys usage by the **model name the provider reports** (`response_metadata["model_name"]`). `get_price` matches that bare name to the `groq:` row of the table. Unknown models get `cost_usd = None`, not a wrong number.
- **This is how Project 01 computes "total tokens" and "estimated cost" per configuration.**

---
## Part 4 · Groq's rate limits

Groq limits each **organization** (not each key or user) on several axes at once; whichever you hit first stops you ([Groq rate limits](https://console.groq.com/docs/rate-limits)): **RPM/RPD** (requests per minute/day), **TPM/TPD** (tokens per minute/day), plus audio limits. Your exact numbers are on the [Limits page](https://console.groq.com/settings/limits). Lesson 00.07 recorded the free tier for `gpt-oss` models as about **8K TPM and 200K TPD** (verified 2026-10-11): roughly 10–20 calls per minute at the sizes in this module.

Every response carries headers that tell you where you stand:

| Header | Refers to |
|---|---|
| `x-ratelimit-limit-requests` / `-remaining-requests` / `-reset-requests` | Requests **per day** |
| `x-ratelimit-limit-tokens` / `-remaining-tokens` / `-reset-tokens` | Tokens **per minute** |
| `retry-after` | Seconds to wait, **only** on a 429 |

LangChain's `AIMessage` doesn't expose HTTP headers, so we read them with the Groq SDK's raw-response API (`with_raw_response`).

In [ ]:
# 💸 Calls Groq once through the raw SDK (fast model), ~100 tokens. Shows rate-limit headers.
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    import groq

    client = groq.Groq(max_retries=0, timeout=30)
    raw = client.chat.completions.with_raw_response.create(
        model=settings.model_fast, messages=[{"role": "user", "content": "Say OK."}], max_completion_tokens=50)
    for name in sorted(h for h in raw.headers if h.startswith("x-ratelimit")):
        print(f"{name:<32} {raw.headers[name]}")
    print("answer:", raw.parse().choices[0].message.content)

**You should now see** six `x-ratelimit-*` headers. `remaining-tokens` is your per-minute token budget right now; `reset-tokens` (e.g. `2.6s`) is when it refills. A careful client slows down when `remaining` gets low instead of waiting for a 429.

---
## Part 5 · Client-side throttling with `InMemoryRateLimiter`

`InMemoryRateLimiter` is a **token bucket** (here "token" means *permission to send a request*, not an LLM token):

| Parameter | Meaning |
|---|---|
| `requests_per_second` | Refill rate. `0.2` = one request every 5 s |
| `check_every_n_seconds` | How often a waiting call checks the bucket |
| `max_bucket_size` | Maximum burst: how many requests can go immediately after a quiet period |

Pass it as `rate_limiter=` to any chat model (`get_model(..., rate_limiter=...)` forwards it). Share **one instance** between models to limit them together. Watch it work on a fake model.

In [ ]:
# 🧪 Offline: 6 calls through a limiter that allows 2 requests/second (bucket of 1 = no burst).
import time

from langchain_core.rate_limiters import InMemoryRateLimiter

limiter = InMemoryRateLimiter(requests_per_second=2, check_every_n_seconds=0.05, max_bucket_size=1)
throttled = GenericFakeChatModel(messages=iter([AIMessage("ok")] * 6), rate_limiter=limiter)

start = time.perf_counter()
for i in range(6):
    throttled.invoke("hi")
    print(f"call {i + 1} at {time.perf_counter() - start:4.2f}s")

### What just happened?

Calls came out about **0.5 s apart** (2 per second), even though the fake model answers instantly. Even call 1 waited 0.5 s: a new limiter starts with an empty bucket. The limiter makes the caller **wait** before sending; nothing fails.

**What it can't do** (the [Models page](https://docs.langchain.com/oss/python/langchain/models#rate-limiting) warns about the first one):

1. It limits **requests**, not **tokens**. One huge prompt still blows your TPM.
2. It lives **in memory, in one process**. Two notebooks, or four server workers, each have their own bucket.
3. It doesn't read Groq's headers; it only follows the rate you configured.

**Picking a rate for the free tier:** if a call uses ~800 tokens and you have 8K TPM, that's 10 calls/min ≈ **0.16 requests/second**. Project 01 uses this calculation.

---
## Part 6 · When you hit a 429 anyway

**Layer 1: the built-in retries.** `max_retries` (default 2 for `ChatGroq`) is passed to the Groq SDK client. What does the SDK do between attempts? Don't guess, read it.

In [ ]:
# 🧪 Offline: print the Groq SDK's retry-delay logic from the installed package.
import inspect

import groq._base_client as base_client
from groq import _constants

import textwrap

source = textwrap.dedent(inspect.getsource(base_client.BaseClient._calculate_retry_timeout))
print(source)
print("\nINITIAL_RETRY_DELAY =", _constants.INITIAL_RETRY_DELAY, "| MAX_RETRY_DELAY =", _constants.MAX_RETRY_DELAY)

### What just happened? (read the printed code)

- If the server sent `retry-after` between 0 and 60 seconds, the SDK **waits exactly that long**. Groq sends it on every 429.
- Otherwise: **exponential backoff** `0.5 × 2ⁿ` seconds, capped at **8 s**, with **jitter** (randomly up to 25% shorter) so many clients don't retry in lockstep.
- So `ChatGroq(max_retries=2)` already handles a brief 429 well. On the free tier a minute-long TPM wait can outlast 2 retries; raise `max_retries` for batch jobs (Lesson 01.03 used 6).

**Layer 2: `with_retry`.** Any `Runnable` can be wrapped to retry on chosen exceptions, with exponential jitter. It's provider-independent, so it also works around chains, tools and custom code. We test it offline with a fake error that looks like an SDK 429.

In [ ]:
# 🧪 Offline: a flaky step that raises a 429-like error twice, then succeeds.
from langchain_core.runnables import RunnableLambda

from mastery.models import explain_model_error


class FakeRateLimitError(Exception):
    """Looks like an SDK 429: has status_code and response.headers['retry-after']."""

    status_code = 429

    def __init__(self) -> None:
        super().__init__("Rate limit reached (simulated)")
        self.response = type("Response", (), {"headers": {"retry-after": "0.2"}})()


attempts = {"n": 0}

def flaky(question: str) -> str:
    attempts["n"] += 1
    if attempts["n"] <= 2:
        raise FakeRateLimitError()
    return f"answered on attempt {attempts['n']}"

print(explain_model_error(FakeRateLimitError()))

In [ ]:
# 🧪 Offline: with_retry retries only the exceptions you list, with exponential jitter between attempts.
resilient = RunnableLambda(flaky).with_retry(
    retry_if_exception_type=(FakeRateLimitError,), stop_after_attempt=4, wait_exponential_jitter=True,
)
start = time.perf_counter()
print(resilient.invoke("hi"), f"after {time.perf_counter() - start:.1f}s")

**Layer 3: your own loop**, when you want to honor `retry-after` and log each wait. Exponential backoff with "full jitter": wait a random time between 0 and `base × 2^attempt`, capped, unless the server told you exactly how long.

In [ ]:
# 🧪 Offline: a small, explicit backoff loop that respects retry-after.
import random


def call_with_backoff(fn, *args, max_attempts: int = 5, base: float = 0.5, cap: float = 20.0):
    """Call fn(*args); on retryable errors wait (retry-after or jittered exponential) and retry."""
    for attempt in range(max_attempts):
        try:
            return fn(*args)
        except Exception as error:
            info = explain_model_error(error)
            if not info.retryable or attempt == max_attempts - 1:
                raise
            wait = info.retry_after if info.retry_after is not None else random.uniform(0, min(cap, base * 2**attempt))
            print(f"attempt {attempt + 1}: {info.kind} → waiting {wait:.2f}s")
            time.sleep(wait)


attempts["n"] = 0
print(call_with_backoff(flaky, "hi"))

### What just happened?

- `with_retry` survived two simulated 429s and answered on attempt 3.
- `call_with_backoff` printed `rate_limit → waiting 0.20s` twice: it used the server's `retry-after` instead of guessing. Non-retryable errors (401, 404, 400) are raised **immediately**: retrying them only wastes time.
- **Don't stack layers blindly.** SDK retries × `with_retry` attempts × your loop = many more requests than you think. Pick one main layer per call path.

---
## Part 7 · Budgeting a workload before you run it

Before a big job, multiply it out. Project 01 runs 10 extraction calls per model configuration.

In [ ]:
# 🧪 Offline: estimated paid-tier cost of a workload, for every model in the price table.
from mastery.pricing import PRICES

CALLS = 10_000                       # e.g. a month of extractions
AVG_INPUT, AVG_OUTPUT = 600, 350     # tokens per call (measure yours with usage_metadata)

rows = [{"model": key, "status": price.status,
         "cost_usd": round(CALLS * (AVG_INPUT * price.input_per_m + AVG_OUTPUT * price.output_per_m) / 1e6, 2),
         "verified": price.verified} for key, price in PRICES.items()]
display(pd.DataFrame(rows).sort_values("cost_usd"))

### What just happened?

For 10,000 calls of ~950 tokens, `gpt-oss-20b` costs about **$1.50**, `gpt-6.1-sol` about **$47**: a 30× spread for the same workload. Whether the expensive model is worth it is an **accuracy** question, which is what Project 01 answers with data.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `usage_metadata` is `None` | Provider didn't report usage (e.g. OpenAI-compatible endpoint streaming without `stream_usage`) | Check `response_metadata`; for `ChatOpenAI` set `stream_usage=True` (Lesson 01.05) |
| Cost table shows `None` | Model name not in `PRICES` (or ambiguous) | Add a dated row to `mastery/pricing.py` with its source |
| Many 429s despite `InMemoryRateLimiter` | Big prompts (TPM), or several processes | Lower the rate, shrink prompts, use one shared limiter per process |
| Batch job dies after 2 retries | TPM wait longer than the SDK's retries | Raise `max_retries` for batch jobs, slow down with a limiter |
| "Latency" is seconds even for a tiny prompt | You timed around `invoke` on a model with `rate_limiter=`: the limiter waits **inside** the call, and so do SDK retry sleeps (LangSmith's run time includes them too) | Keep the limiter off the model and wait for it before starting the clock (`mastery.evals.timed_invoke(..., limiter=...)`); compare with the server time Groq reports in `response_metadata["token_usage"]` |
| 401 retried in a loop | Retrying non-retryable errors | Retry only `explain_model_error(e).retryable` |
| Answer empty, `finish_reason="length"` | Reasoning used up `max_tokens` | Lower `reasoning_effort` or raise `max_tokens` |

## ⚠️ Production notes

> ⚠️ **Production note: in-memory rate limiting.**
> `InMemoryRateLimiter` only coordinates threads in one process. In production with several workers: use a shared limiter (e.g. Redis-based), limit by tokens as well as requests, read the provider's `x-ratelimit-*` headers to adapt, and queue work instead of failing when the budget is exhausted.

> ⚠️ **Production note: estimated costs.**
> `mastery.pricing` is a hand-maintained table. In production: reconcile with the provider's billing data, track cost per feature/customer via trace metadata (LangSmith cost tracking), and set budget alerts in the provider console.

## 🏋️ Exercises

1. **(Easy)** Run Part 2 with `reasoning_effort` set to `"low"`, `"medium"` and `"high"`. Tabulate output and reasoning tokens. Which setting would you pay for, for this question?
2. **(Easy)** Change the limiter in Part 5 to `requests_per_second=1, max_bucket_size=3`. Predict the timing of 6 calls (hint: burst first), then check.
3. **(Medium)** Write `budget_rate(tpm: int, tokens_per_call: int, safety: float = 0.8) -> float` that returns a `requests_per_second` for `InMemoryRateLimiter`. What does it return for 8,000 TPM and 800 tokens per call?
4. **(Stretch)** Combine Parts 3 and 5: batch the 12 tickets from Lesson 01.03 through `get_model("fast", rate_limiter=...)` with the rate from exercise 3, inside `get_usage_metadata_callback()`, and print the cost table and total wall time. Did you get any 429s?

<details>
<summary><b>Show solutions</b></summary>

**2.** Surprise: a **new** limiter's bucket starts **empty**, so the 6 calls come out at about 1, 2, 3, 4, 5, 6 s; no burst. The burst appears after a quiet period: wait 3+ seconds and call again, and the next 3 calls go out immediately (the bucket refilled to `max_bucket_size=3`), then one per second. (We verified this on `langchain-core` 1.6.9.) Lesson: test a component's behavior instead of assuming it from parameter names.

**3.**
```python
def budget_rate(tpm: int, tokens_per_call: int, safety: float = 0.8) -> float:
    """requests_per_second that keeps usage under `safety` x the tokens-per-minute limit."""
    return tpm * safety / tokens_per_call / 60

print(round(budget_rate(8_000, 800), 3))   # 0.133 → about one call every 7.5 s
```
</details>

## ✅ Checkpoint

- [ ] I can read `usage_metadata` and say which tokens are billed as output.
- [ ] I aggregated usage across models and produced a cost table with `mastery.pricing.cost_report`.
- [ ] I can explain Groq's limit types and read its rate-limit headers.
- [ ] I throttled calls with `InMemoryRateLimiter` and know its three limitations.
- [ ] I can explain what the Groq SDK does on a 429 (from its code) and wrote a backoff that respects `retry-after`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open your own [Groq Limits page](https://console.groq.com/settings/limits). Record your **actual** RPM, RPD, TPM and TPD for the fast and reasoning models, compare them with the [rate limits docs](https://console.groq.com/docs/rate-limits) and the headers from Part 4, and compute a safe `requests_per_second` for Project 01 with exercise 3. Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Tokens drive cost, latency and limits. Reasoning tokens are output tokens.
- `usage_metadata` per call, `get_usage_metadata_callback` per job, `mastery.pricing` for dollars.
- Concurrency (01.05) ≠ rate (`InMemoryRateLimiter`) ≠ recovery (retries with backoff and `retry-after`).
- Read the SDK to know what "retries" really do; retry only retryable errors.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 01.07 · Under the hood: Runnables and LCEL (legacy-aware)** → [`01-07-runnables-and-lcel.ipynb`](01-07-runnables-and-lcel.ipynb).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)